# TP Pinecone : réorganisation (reranking) et recherche sémantique sur des notes médicales

Ce notebook complète le TP en trois grandes étapes : tester le modèle de réorganisation de Pinecone sur un petit exemple, construire un index serverless pour des notes médicales, puis combiner recherche sémantique et reranking sur ces notes.

## Partie 1 : Charger les documents et exécuter le modèle de réorganisation

### 1. Installer les bibliothèques Pinecone

In [ ]:
!pip install -U pinecone==6.0.1 pinecone-notebooks


### 2. S'authentifier auprès de Pinecone

In [ ]:
import os

if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()


### 3. Instancier le client Pinecone

In [ ]:
from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)


### 4. Définir la requête et les documents

On mélange volontairement des documents sur « Apple » l'entreprise et « apple » le fruit, pour vérifier que le reranking sait distinguer les deux sens selon le contexte de la requête.

In [ ]:
query = "Tell me about Apple\'s products"

documents = [
    "The apple is a sweet, edible fruit produced by an apple tree, and it is one of the most widely cultivated fruits in the world.",
    "Apple Inc. designs and sells consumer electronics such as the iPhone, iPad, and MacBook.",
    "Apples are rich in fiber and vitamin C, and they come in varieties like Granny Smith and Fuji.",
    "Apple's latest product lineup includes updated versions of the Apple Watch and AirPods.",
    "Apple pie is a traditional dessert made by baking sliced apples inside a pastry crust."
]


### 5. Appeler le service de réorganisation (reranking)

In [ ]:
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)


### 6. Examiner les résultats réordonnés

L'objet renvoyé par `pc.inference.rerank()` expose ses résultats via l'attribut `.data` : une liste d'objets qui possèdent chacun `.score` et `.document.text`.

In [ ]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"{i+1}. score={m.score:.4f} | {m.document.text}")

show_reranked_results(query, reranked.data)


## Partie 2 : Mise en place d'un index serverless pour les notes médicales

### 1. Installer les bibliothèques de données et de modèles

In [ ]:
!pip install pandas torch transformers


### 2. Importer les modules et définir les paramètres d'environnement

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Paramètres cloud/région (valeurs par défaut qui conviennent à la plupart des comptes)
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

# Spécification serverless
spec = ServerlessSpec(cloud=cloud, region=region)

# Nom de l'index
index_name = 'medical-notes-index'


### 3. Créer (ou recréer) l'index

Le modèle d'embedding utilisé plus loin (`all-MiniLM-L6-v2`) produit des vecteurs de dimension 384 ; la métrique cosinus est adaptée à des embeddings de texte.

In [ ]:
# Supprime un éventuel index existant portant le même nom
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Crée le nouvel index
pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec
)


## Partie 3 : Charger les données d'exemple

### 1. Télécharger et lire le fichier JSONL

In [ ]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    url = "https://raw.githubusercontent.com/pineconeio/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)


### 2. Prévisualiser le DataFrame

In [ ]:
print("Data shape:", df.shape)  # (nombre de lignes, nombre de colonnes)
df.head()


## Partie 4 : Mise à jour des données dans l'index

### 1. Instancier le client d'index et effectuer la mise à jour

In [ ]:
# Instancie un client pour l'index
index = pc.Index(name=index_name)

# Insère (upsert) les données du DataFrame dans l'index
index.upsert_from_dataframe(df)


### 2. Attendre que l'index soit disponible

In [ ]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()


## Partie 5 : Fonction de requête et d'intégration (embedding)

### 1. Définir la fonction d'intégration

`model_output.last_hidden_state[0]` a pour forme `(longueur_de_séquence, taille_cachée)` une fois le batch retiré : on moyenne donc sur la dimension 0 (la longueur de séquence) pour obtenir un seul vecteur par phrase.

In [ ]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)

    embedding = model_output.last_hidden_state[0].mean(dim=0)
    return embedding


### 2. Exécuter une requête de recherche sémantique

In [ ]:
# Construire une requête à rechercher
question = "What are the treatment options for a patient with chest pain?"
query = get_embedding(question).tolist()

# Récupérer les résultats
results = index.query(vector=[query], top_k=10, include_metadata=True)

# Trier les résultats par score décroissant
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)


## Partie 6 : Afficher et réorganiser les notes cliniques

### 1. Afficher les premiers résultats de la recherche

In [ ]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f'      Score: {match["score"]}')
        print(f'      Metadata: {match["metadata"]}')
        print('')

show_results(question, sorted_matches)


### 2. Préparer les documents pour le réexamen (reranking)

In [ ]:
# Crée un document par résultat, avec un champ "reranking_field" qui concatène les métadonnées
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]


### 3. Exécuter le réordonnancement serverless

In [ ]:
# Requête plus précise pour le reranking
refined_query = "patient needing knee surgery"

# Réordonnancement basé sur cette requête et sur le champ "reranking_field"
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)


### 4. Afficher les résultats réorganisés

Comme en Partie 1, les résultats sont accessibles via `.data`. Chaque élément possède `.score` (le nouveau score de pertinence) et `.document.reranking_field` (le champ texte utilisé pour le reclassement).

In [ ]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f'      Score: {match.score}')
        print(f'      Reranking Field: {match.document.reranking_field}')
        print('')

show_reranked_results(refined_query, reranked_results.data)


### 5. Nettoyage (facultatif)

In [ ]:
# Supprime l'index pour éviter des frais inutiles une fois le TP terminé
pc.delete_index(name=index_name)
